# Fase 3: Hough con memoria constante en Google Colab

Compila y valida `src/houghConst.cu` (memoria global + constante) junto con la versión global, en la misma sesión y GPU.

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4.**
2. En tu computadora ejecuta `make colab-package` y sube `build/proyecto-colab.zip` cuando se solicite.
3. Ejecuta las celdas en orden y guarda el archivo descargado en `results/constante-validacion.txt`.

In [ ]:
!nvidia-smi
!nvcc --version

## Archivos del proyecto

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os
uploaded = files.upload()
assert len(uploaded) == 1, "Sube solamente proyecto-colab.zip"
root = Path('/content/proyecto-hough')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        assert (root / name).resolve().is_relative_to(root.resolve()), "Ruta ZIP invalida"
    archive.extractall(root)
os.chdir(root)
assert Path('src/houghConst.cu').exists(), "El ZIP no contiene src/houghConst.cu"
print('Proyecto:', root)

## Compilación

La versión global se compila con el Makefile. La constante usa las mismas opciones de `nvcc` y el mismo `pgm.o`, sin modificar el Makefile compartido.

In [ ]:
import subprocess
subprocess.run(['make'], check=True)
subprocess.run(['nvcc', '-I.', '-O2', '-std=c++17', '--fmad=false',
                'src/houghConst.cu', 'build/pgm.o', '-o', 'build/hough_const'], check=True)
print('Compilados: build/hough y build/hough_const')

## Validación y evidencia

Ejecuta ambas versiones con la imagen original y la variante constante con una imagen de 21 píxeles (bloque parcial), una imagen negra y entradas inválidas. `cuobjdump -res-usage` muestra la memoria constante que usa cada kernel. El log se guarda y descarga aunque alguna prueba falle.

In [ ]:
import tempfile
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
fallas = []

def ejecutar(etiqueta, comando, debe_pasar=True):
    run = subprocess.run(comando, text=True, capture_output=True)
    salida = f"== {etiqueta}: {' '.join(comando)}\n{run.stdout}{run.stderr}"
    print(salida)
    if (run.returncode == 0) != debe_pasar:
        fallas.append(etiqueta)
    return salida

for programa in ['./build/hough', './build/hough_const']:
    log += ejecutar('original', [programa, 'data/runway.pgm'])
with tempfile.TemporaryDirectory() as directory:
    for name, data in [('parcial', bytes([255]) * 21), ('negra', bytes(21))]:
        path = Path(directory) / (name + '.pgm')
        path.write_bytes(b'P5\n7 3\n255\n' + data)
        log += ejecutar(name, ['./build/hough_const', str(path)])
for args in [[], ['archivo-inexistente.pgm']]:
    log += ejecutar('entrada invalida', ['./build/hough_const', *args], debe_pasar=False)
for programa in ['build/hough', 'build/hough_const']:
    try:
        log += ejecutar('recursos del kernel', ['cuobjdump', '-res-usage', programa])
    except FileNotFoundError:
        log += 'cuobjdump no disponible\n'

log += 'RESULTADO: ' + ('FALLAS en ' + ', '.join(fallas) if fallas else 'todas las pruebas pasaron') + '\n'
Path('results').mkdir(parents=True, exist_ok=True)
Path('results/constante-validacion.txt').write_text(log)
print('Evidencia: results/constante-validacion.txt')
files.download('results/constante-validacion.txt')
assert not fallas, 'Fallaron: ' + ', '.join(fallas)